In [1]:
import chess
import chess.engine
import chess.pgn
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import io, os

STOCKFISH_PATH = r"E:\chess-predictor\stockfish\stockfish-windows-x86-64-universal.exe"

print("python-chess:", chess.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("Stockfish path exists:", os.path.exists(STOCKFISH_PATH))
print("✅ Everything loaded")

python-chess: 1.11.2
pandas: 3.0.6
numpy: 2.5.3
Stockfish path exists: True
✅ Everything loaded


In [2]:
import os

pgn_file = "mega2600_part_01.pgn"

if os.path.exists(pgn_file):
    size_mb = os.path.getsize(pgn_file) / (1024 * 1024)
    print(f"{pgn_file} found — {size_mb:.1f} MB")
else:
    print(f"❌ {pgn_file} not found")
    print("Files in current folder:", os.listdir("."))

mega2600_part_01.pgn found — 25.0 MB


In [3]:
with open(pgn_file, "r", encoding="utf-8", errors="ignore") as f:
    preview = f.read(1500)

print(preview)

[Event "GBR-ch 98th"]
[Site "Sheffield"]
[Date "2011.07.29"]
[Round "5"]
[White "Adams, Michael"]
[Black "Jones, Gawain C"]
[Result "1/2-1/2"]
[WhiteElo "2715"]
[BlackElo "2606"]
[ECO "B76"]
[EventDate "2011.07.24"]
[PlyCount "84"]
[EventType "swiss"]
[EventRounds "11"]
[EventCountry "GBR"]
[Source "John Saunders"]
[SourceDate "2015.07.24"]

1. e4 c5 2. Nf3 d6 3. d4 cxd4 4. Nxd4 Nf6 5. Nc3 g6 6. Be3 Bg7 7. f3 O-O 8.
Qd2 Nc6 9. O-O-O d5 10. exd5 Nxd5 11. Nxc6 bxc6 12. Bd4 Bxd4 13. Qxd4 Qb6
14. Na4 Qa5 15. b3 Be6 16. Bc4 Rfd8 17. Qc5 Qc7 18. Kb1 Nf4 19. g3 Bxc4 20.
Qxc4 Nd5 21. Rhe1 Nb6 22. Qe4 Nxa4 23. Qxa4 e6 24. Qe4 Rd5 25. c4 Rd7 26.
Rxd7 Qxd7 27. Kc2 Rd8 28. Qe2 e5 29. Rd1 Qxd1+ 30. Qxd1 Rxd1 31. Kxd1 c5
32. a3 a5 33. b4 cxb4 34. axb4 axb4 35. c5 Kf8 36. Kc2 Ke7 37. Kb3 Kd7 38.
Kxb4 Kc6 39. Kc4 h6 40. f4 e4 41. Kd4 f5 42. Kc4 e3 1/2-1/2

[Event "GBR-ch 98th"]
[Site "Sheffield"]
[Date "2011.07.29"]
[Round "5"]
[White "Short, Nigel D"]
[Black "Howell, David"]
[Result "1/2-1/2"]
[White

In [4]:
import chess.pgn
import pandas as pd

pgn_file = "mega2600_part_01.pgn"
max_games = 200

games_list = []

# Open the file and read one game at a time
with open(pgn_file, encoding="utf-8", errors="ignore") as f:
    for i in range(max_games):
        game = chess.pgn.read_game(f)
        if game is None:
            break  # We ran out of games

        headers = game.headers
        result_str = headers.get("Result", "*")

        # Skip games with no result
        if result_str == "1-0":
            result = 1      # White wins
        elif result_str == "0-1":
            result = -1     # Black wins
        elif result_str == "1/2-1/2":
            result = 0      # Draw
        else:
            continue        # Skip unfinished

        # Get ELOs (some games might not have them)
        try:
            white_elo = int(headers.get("WhiteElo", 0))
            black_elo = int(headers.get("BlackElo", 0))
        except ValueError:
            white_elo = 0
            black_elo = 0

        # Count moves
        moves = list(game.mainline_moves())

        games_list.append({
            "White": headers.get("White"),
            "Black": headers.get("Black"),
            "WhiteElo": white_elo,
            "BlackElo": black_elo,
            "Result": result,
            "NumMoves": len(moves),
            "ECO": headers.get("ECO", ""),
        })

# Turn the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(games_list)

print(f"Loaded {len(df)} games.")
print(f"Shape: {df.shape}")
print()
print(df.head())

Loaded 200 games.
Shape: (200, 7)

             White            Black  WhiteElo  BlackElo  Result  NumMoves  ECO
0   Adams, Michael  Jones, Gawain C      2715      2606       0        84  B76
1   Short, Nigel D    Howell, David      2687      2625       0       151  C65
2    Howell, David   Adams, Michael      2625      2715      -1       108  C45
3  Jones, Gawain C    Howell, David      2606      2625       1        79  C54
4   Short, Nigel D   Adams, Michael      2687      2715       0        91  C50


In [5]:
print("Total games:", len(df))
print()
print("Result distribution (1=White, 0=Draw, -1=Black):")
print(df["Result"].value_counts())
print()
print("Average White ELO:", int(df["WhiteElo"].mean()))
print("Average Black ELO:", int(df["BlackElo"].mean()))
print()
print("Average game length (moves):", int(df["NumMoves"].mean()))

Total games: 200

Result distribution (1=White, 0=Draw, -1=Black):
Result
 0    118
 1     43
-1     39
Name: count, dtype: int64

Average White ELO: 2644
Average Black ELO: 2646

Average game length (moves): 76


In [6]:
import chess
import io

# The value of each piece in "pawns"
PIECE_VALUES = {
    chess.PAWN: 1,
    chess.KNIGHT: 3,
    chess.BISHOP: 3,
    chess.ROOK: 5,
    chess.QUEEN: 9,
    chess.KING: 0,
}

def features_at_ply(board):
    """Given a chess board, return a dict of numeric features."""
    mat_white = 0
    mat_black = 0
    for square in chess.SQUARES:
        piece = board.piece_at(square)
        if piece is None:
            continue
        if piece.color == chess.WHITE:
            mat_white += PIECE_VALUES[piece.piece_type]
        else:
            mat_black += PIECE_VALUES[piece.piece_type]

    return {
        "material_balance": mat_white - mat_black,
        "white_bishops": len(board.pieces(chess.BISHOP, chess.WHITE)),
        "black_bishops": len(board.pieces(chess.BISHOP, chess.BLACK)),
        "total_pieces": len(board.piece_map()),
    }

def extract_features_from_game(game, snapshot_ply=80):
    """Replay the game to snapshot_ply and extract features."""
    board = game.board()
    moves = list(game.mainline_moves())

    for i, move in enumerate(moves):
        if i >= snapshot_ply:
            break
        board.push(move)

    feats = features_at_ply(board)
    feats["ply"] = min(snapshot_ply, len(moves))
    return feats

# Re-read the games and extract features
features_list = []

with open(pgn_file, encoding="utf-8", errors="ignore") as f:
    for i in range(200):
        game = chess.pgn.read_game(f)
        if game is None:
            break

        headers = game.headers
        result_str = headers.get("Result", "*")

        if result_str == "1-0":
            result = 1
        elif result_str == "0-1":
            result = -1
        elif result_str == "1/2-1/2":
            result = 0
        else:
            continue

        try:
            white_elo = int(headers.get("WhiteElo", 0))
            black_elo = int(headers.get("BlackElo", 0))
        except ValueError:
            white_elo = 0
            black_elo = 0

        feats = extract_features_from_game(game, snapshot_ply=80)  # 80 plies = 40 moves
        feats["white_elo"] = white_elo
        feats["black_elo"] = black_elo
        feats["elo_diff"] = white_elo - black_elo
        feats["result"] = result

        features_list.append(feats)

features_df = pd.DataFrame(features_list)
print(f"Extracted features for {len(features_df)} games.")
print()
print(features_df.head())

Extracted features for 200 games.

   material_balance  white_bishops  black_bishops  total_pieces  ply  \
0                 0              0              0            10   80   
1                 1              1              2            19   80   
2                -1              0              0            15   80   
3                 1              0              0            13   79   
4                 1              1              0            13   80   

   white_elo  black_elo  elo_diff  result  
0       2715       2606       109       0  
1       2687       2625        62       0  
2       2625       2715       -90      -1  
3       2606       2625       -19       1  
4       2687       2715       -28       0  


In [7]:
print("Feature table shape:", features_df.shape)
print()
print("Material balance stats:")
print(features_df["material_balance"].describe())
print()
print("Correlation between material balance and result:")
print(features_df[["material_balance", "elo_diff", "result"]].corr())

Feature table shape: (200, 9)

Material balance stats:
count    200.000000
mean      -0.020000
std        1.832043
min       -9.000000
25%        0.000000
50%        0.000000
75%        1.000000
max        6.000000
Name: material_balance, dtype: float64

Correlation between material balance and result:
                  material_balance  elo_diff    result
material_balance          1.000000  0.085752  0.295322
elo_diff                  0.085752  1.000000  0.182341
result                    0.295322  0.182341  1.000000


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Features we'll feed the model
feature_cols = [
    "material_balance",
    "white_bishops",
    "black_bishops",
    "total_pieces",
    "ply",
    "elo_diff",
]

X = features_df[feature_cols]
y = features_df["result"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} games")
print(f"Testing set:  {len(X_test)} games")
print()

# Train the model
clf = RandomForestClassifier(
    n_estimators=200,      # 200 decision trees
    max_depth=6,           # limit depth to avoid overfitting on small data
    random_state=42,
)
clf.fit(X_train, y_train)

# Test it
predictions = clf.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Test accuracy: {accuracy:.1%}")
print()
print(classification_report(y_test, predictions, 
                            target_names=["Black win (-1)", "Draw (0)", "White win (1)"]))

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# ⬇️ NEW: added stockfish_eval to the list
feature_cols = [
    "material_balance",
    "white_bishops",
    "black_bishops",
    "total_pieces",
    "ply",
    "elo_diff",
    "stockfish_eval",       # ← the new feature
]

X = features_df[feature_cols]
y = features_df["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} games")
print(f"Testing set:  {len(X_test)} games")
print()

clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42,
)
clf.fit(X_train, y_train)

predictions = clf.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Test accuracy: {accuracy:.1%}")
print()
print(classification_report(y_test, predictions,
                            target_names=["Black win (-1)", "Draw (0)", "White win (1)"]))

Training set: 160 games
Testing set:  40 games

Test accuracy: 70.0%

                precision    recall  f1-score   support

Black win (-1)       0.67      0.50      0.57         8
      Draw (0)       0.73      0.92      0.81        24
 White win (1)       0.50      0.25      0.33         8

      accuracy                           0.70        40
     macro avg       0.63      0.56      0.57        40
  weighted avg       0.67      0.70      0.67        40



KeyError: "['stockfish_eval'] not in index"

In [ ]:
import matplotlib.pyplot as plt

importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]

print("Feature importance (which numbers matter most):")
for i in indices:
    print(f"  {feature_cols[i]:20s}  {importances[i]:.3f}")

# Plot it
plt.figure(figsize=(8, 4))
plt.bar(range(len(feature_cols)), importances[indices])
plt.xticks(range(len(feature_cols)), [feature_cols[i] for i in indices], rotation=45, ha="right")
plt.title("Feature importance")
plt.tight_layout()
plt.show()

In [ ]:
from stockfish import Stockfish

sf = Stockfish(
    path=r"E:\chess-predictor\stockfish\stockfish-windows-x86-64-universal.exe",
    depth=10,
    parameters={"Threads": 2, "Hash": 128},
)

sf.set_fen_position("rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1")
print("Best move:", sf.get_best_move())
print("Eval:", sf.get_evaluation())

In [17]:
from stockfish import Stockfish

sf = Stockfish(
    path=r"E:\chess-predictor\stockfish\stockfish-windows-x86-64-universal.exe",
    depth=15,
    parameters={"Threads": 2, "Hash": 128},
)

sf.set_fen_position("rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1")
print("Best move:", sf.get_best_move())
print("Eval:", sf.get_evaluation())

Best move: e2e4
Eval: {'type': 'cp', 'value': 26}


In [ ]:
from stockfish import Stockfish
import time

sf = Stockfish(
    path=r"E:\chess-predictor\stockfish\stockfish-windows-x86-64-universal.exe",
    depth=10,
    parameters={"Threads": 2, "Hash": 128},
)

print("Re-reading games and evaluating each position at ply 80...")
start = time.time()
sf_evals = []

with open(pgn_file, encoding="utf-8", errors="ignore") as f:
    for i in range(200):
        game = chess.pgn.read_game(f)
        if game is None:
            break
        headers = game.headers
        if headers.get("Result", "*") not in ("1-0", "0-1", "1/2-1/2"):
            continue

        # Replay to ply 80
        board = game.board()
        for j, mv in enumerate(game.mainline_moves()):
            if j >= 80:
                break
            board.push(mv)

        # Ask Stockfish about this position
        sf.set_fen_position(board.fen())
        evaluation = sf.get_evaluation()

        # Convert to centipawns from White's perspective
        if evaluation["type"] == "cp":
            cp = evaluation["value"]
        elif evaluation["type"] == "mate":
            cp = 10000 if evaluation["value"] > 0 else -10000
        else:
            cp = 0

        # Negate if Black is to move (FEN's perspective is side-to-move)
        if board.turn == chess.BLACK:
            cp = -cp

        sf_evals.append(cp)

        if (i + 1) % 25 == 0:
            print(f"  {i+1}/200 done — {time.time()-start:.1f}s elapsed")

print(f"\nDone. Evaluated {len(sf_evals)} positions in {time.time()-start:.1f}s")

features_df["stockfish_eval"] = sf_evals
print()
print(features_df[["stockfish_eval", "result"]].head(10))

In [ ]:
def predict_game(pgn_string, snapshot_ply=80, verbose=True):
    """
    Take a PGN string, analyze it at ply 80, return prediction.
    """
    # 1. Parse the game
    game = chess.pgn.read_game(io.StringIO(pgn_string))
    if game is None:
        return {"error": "Could not parse PGN"}

    headers = game.headers
    white_elo = int(headers.get("WhiteElo", 0) or 0)
    black_elo = int(headers.get("BlackElo", 0) or 0)

    # 2. Replay to snapshot_ply
    board = game.board()
    for i, mv in enumerate(game.mainline_moves()):
        if i >= snapshot_ply:
            break
        board.push(mv)

    # 3. Extract handcrafted features
    feats = features_at_ply(board)
    feats["ply"] = min(snapshot_ply, len(list(game.mainline_moves())))
    feats["elo_diff"] = white_elo - black_elo

    # 4. Stockfish eval
    sf.set_fen_position(board.fen())
    ev = sf.get_evaluation()
    if ev["type"] == "cp":
        cp = ev["value"]
    elif ev["type"] == "mate":
        cp = 10000 if ev["value"] > 0 else -10000
    else:
        cp = 0
    if board.turn == chess.BLACK:
        cp = -cp
    feats["stockfish_eval"] = cp

    # 5. Predict with the trained model
    X_new = pd.DataFrame([feats])[feature_cols]
    proba = clf.predict_proba(X_new)[0]
    pred_class = clf.classes_[np.argmax(proba)]

    # 6. Format output
    label = {1: "White wins", 0: "Draw", -1: "Black wins"}[pred_class]
    confidence = max(proba)
    probs = dict(zip(clf.classes_, proba))

    if verbose:
        print(f"White: {headers.get('White')} ({white_elo})")
        print(f"Black: {headers.get('Black')} ({black_elo})")
        print(f"Position at ply {feats['ply']} — material balance: {feats['material_balance']:+d}")
        print(f"Stockfish eval: {cp:+d} centipawns")
        print()
        print(f"Prediction: {label}")
        print(f"Confidence: {confidence:.1%}")
        print(f"  P(White wins) = {probs.get(1, 0):.2%}")
        print(f"  P(Draw)       = {probs.get(0, 0):.2%}")
        print(f"  P(Black wins) = {probs.get(-1, 0):.2%}")
    else:
        return {"label": label, "confidence": confidence, "probs": probs}

    return label


# Get a full PGN from our file to test with
with open(pgn_file, encoding="utf-8", errors="ignore") as f:
    first_game_text = str(chess.pgn.read_game(f))

print("Testing predict_game() on the first game in our dataset...\n")
predict_game(first_game_text)

In [ ]:
# Find a game with a decisive result to test on
decisive_idx = features_df[features_df["result"] == 1].index[0]  # first White win

# Get that game's PGN text
with open(pgn_file, encoding="utf-8", errors="ignore") as f:
    all_games = []
    for _ in range(decisive_idx + 1):
        g = chess.pgn.read_game(f)
        if g is not None:
            all_games.append(str(g))

test_pgn = all_games[decisive_idx]
print(f"Testing on game #{decisive_idx} (which actually was a White win)\n")
predict_game(test_pgn)

In [ ]:
import joblib

joblib.dump(clf, "chess_model.pkl")
joblib.dump(feature_cols, "feature_cols.pkl")
print("Model saved to chess_model.pkl ✅")

In [ ]:
clf = joblib.load("chess_model.pkl")
feature_cols = joblib.load("feature_cols.pkl")

In [ ]:
# Predict on the test set and find mistakes
test_pred = clf_v2.predict(X_test)

# Align back to original dataframe
test_indices = y_test.index
mistakes = features_df.loc[test_indices][test_pred != y_test.values]

print(f"Model made {len(mistakes)} mistakes out of {len(y_test)}")
print()
print(mistakes[["elo_diff", "material_balance", "stockfish_eval", "result"]])

In [ ]:
# Find the actual game for row 52
row52 = features_df.loc[52]
print("Row 52 features:")
print(row52)
print()

# We need to re-read the PGN file to get the actual game
# Row 52 in features_df corresponds to game index 52 in the file
with open(pgn_file, encoding="utf-8", errors="ignore") as f:
    games = []
    for i in range(200):
        g = chess.pgn.read_game(f)
        if g is None:
            break
        games.append(g)

# Note: game index in the file may not equal row index if some were skipped
# Let's find it by matching ELOs and result
target = games[52]
print(f"Game #{52}:")
print(f"  White: {target.headers.get('White')} ({target.headers.get('WhiteElo')})")
print(f"  Black: {target.headers.get('Black')} ({target.headers.get('BlackElo')})")
print(f"  Result: {target.headers.get('Result')}")
print(f"  ECO: {target.headers.get('ECO')}")
print()
print("Full PGN:")
print(target)

In [ ]:
# Replay to ply 80 and show the board
board = target.board()
moves = list(target.mainline_moves())
for i, mv in enumerate(moves):
    if i >= 80:
        break
    board.push(mv)

print(f"Position after {board.fullmove_number - 1} full moves:")
print(board)
print()
print(f"Side to move: {'White' if board.turn == chess.WHITE else 'Black'}")
print(f"FEN: {board.fen()}")

In [ ]:
# Add a feature that captures disagreement between Stockfish and material
features_df["stockfish_vs_material"] = (
    features_df["stockfish_eval"] - features_df["material_balance"] * 100
)

# Look at the disagreement for row 52 vs others
print(features_df.loc[[31, 52, 185, 198], 
                     ["material_balance", "stockfish_eval", "stockfish_vs_material", "result"]])

In [ ]:
feature_cols_v2 = feature_cols + ["stockfish_vs_material"]  # add to existing list

X = features_df[feature_cols_v2]
y = features_df["result"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf_v2 = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
clf_v2.fit(X_train, y_train)

pred_v2 = clf_v2.predict(X_test)
print(f"Accuracy with new feature: {accuracy_score(y_test, pred_v2):.1%}")
print(classification_report(y_test, pred_v2, target_names=["Black win", "Draw", "White win"]))

In [ ]:
import joblib
joblib.dump(clf_v2, "chess_model_v2.pkl")
joblib.dump(feature_cols_v2, "feature_cols_v2.pkl")

In [ ]:
!pip install streamlit

In [ ]:
import joblib
joblib.dump(clf_v2, "chess_model.pkl")
joblib.dump(feature_cols_v2, "feature_cols.pkl")
print("Model + features saved ✅")

In [1]:
#Cell A
import chess
import chess.pgn
import pandas as pd
import numpy as np
import io, os
from stockfish import Stockfish

# ---------- Config ----------
STOCKFISH_PATH = r"E:\chess-predictor\stockfish\stockfish-windows-x86-64-universal.exe"
PGN_FILE = "mega2600_part_01.pgn"
MAX_GAMES = 200
SNAPSHOT_PLY = 80
SF_DEPTH = 15

# ---------- Open Stockfish once ----------
sf = Stockfish(path=STOCKFISH_PATH, depth=SF_DEPTH, parameters={"Threads": 2, "Hash": 128})

# ---------- Piece values ----------
PIECE_VALUES = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
    chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0,
}

def features_at_ply(board):
    mat_w = mat_b = 0
    for sq in chess.SQUARES:
        p = board.piece_at(sq)
        if p is None: continue
        if p.color == chess.WHITE:
            mat_w += PIECE_VALUES[p.piece_type]
        else:
            mat_b += PIECE_VALUES[p.piece_type]
    return {
        "material_balance": mat_w - mat_b,
        "white_bishops": len(board.pieces(chess.BISHOP, chess.WHITE)),
        "black_bishops": len(board.pieces(chess.BISHOP, chess.BLACK)),
        "total_pieces": len(board.piece_map()),
    }

# ---------- Process all games ----------
rows = []
print(f"Processing {MAX_GAMES} games (ply {SNAPSHOT_PLY}, Stockfish depth {SF_DEPTH})...")

with open(PGN_FILE, encoding="utf-8", errors="ignore") as f:
    for i in range(MAX_GAMES):
        game = chess.pgn.read_game(f)
        if game is None: break

        headers = game.headers
        result_str = headers.get("Result", "*")
        if result_str == "1-0": result = 1
        elif result_str == "0-1": result = -1
        elif result_str == "1/2-1/2": result = 0
        else: continue

        white_elo = int(headers.get("WhiteElo", 0) or 0)
        black_elo = int(headers.get("BlackElo", 0) or 0)

        board = game.board()
        moves = list(game.mainline_moves())
        for j, mv in enumerate(moves):
            if j >= SNAPSHOT_PLY: break
            board.push(mv)

        feats = features_at_ply(board)
        feats["ply"] = min(SNAPSHOT_PLY, len(moves))
        feats["white_elo"] = white_elo
        feats["black_elo"] = black_elo
        feats["elo_diff"] = white_elo - black_elo
        feats["result"] = result

        # Stockfish eval
        sf.set_fen_position(board.fen())
        ev = sf.get_evaluation()
        if ev["type"] == "cp":
            cp = ev["value"]
        elif ev["type"] == "mate":
            cp = 10000 if ev["value"] > 0 else -10000
        else:
            cp = 0
        if board.turn == chess.BLACK:
            cp = -cp

        feats["stockfish_eval"] = cp
        feats["stockfish_vs_material"] = cp - feats["material_balance"] * 100
        rows.append(feats)

features_df = pd.DataFrame(rows)

print(f"✅ features_df built — shape: {features_df.shape}")
print(f"   Columns: {list(features_df.columns)}")
print()
print(features_df.head())

# Save a backup so we never have to rebuild
features_df.to_csv("features_df_backup.csv", index=False)
print()
print("✅ Backup saved to features_df_backup.csv")

Processing 200 games (ply 80, Stockfish depth 15)...
✅ features_df built — shape: (200, 11)
   Columns: ['material_balance', 'white_bishops', 'black_bishops', 'total_pieces', 'ply', 'white_elo', 'black_elo', 'elo_diff', 'result', 'stockfish_eval', 'stockfish_vs_material']

   material_balance  white_bishops  black_bishops  total_pieces  ply  \
0                 0              0              0            10   80   
1                 1              1              2            19   80   
2                -1              0              0            15   80   
3                 1              0              0            13   79   
4                 1              1              0            13   80   

   white_elo  black_elo  elo_diff  result  stockfish_eval  \
0       2715       2606       109       0              -9   
1       2687       2625        62       0               0   
2       2625       2715       -90      -1            -147   
3       2606       2625       -19       1        

In [2]:
#Cell B

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

feature_cols_v2 = [
    "material_balance",
    "white_bishops",
    "black_bishops",
    "total_pieces",
    "ply",
    "elo_diff",
    "stockfish_eval",
    "stockfish_vs_material",
]

X = features_df[feature_cols_v2]
y = features_df["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf_v2 = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
clf_v2.fit(X_train, y_train)

pred_v2 = clf_v2.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, pred_v2):.1%}")
print()
print(classification_report(y_test, pred_v2,
                            target_names=["Black win", "Draw", "White win"]))

Accuracy: 90.0%

              precision    recall  f1-score   support

   Black win       1.00      0.88      0.93         8
        Draw       0.88      0.96      0.92        24
   White win       0.86      0.75      0.80         8

    accuracy                           0.90        40
   macro avg       0.91      0.86      0.88        40
weighted avg       0.90      0.90      0.90        40



In [3]:
#Cell c
print("stockfish_eval range in training data:")
print(f"  min: {features_df['stockfish_eval'].min()}")
print(f"  max: {features_df['stockfish_eval'].max()}")
print()
print("stockfish_vs_material range:")
print(f"  min: {features_df['stockfish_vs_material'].min()}")
print(f"  max: {features_df['stockfish_vs_material'].max()}")

stockfish_eval range in training data:
  min: -10000
  max: 10000

stockfish_vs_material range:
  min: -10100
  max: 10400


In [4]:
features_df = pd.read_csv("features_df_backup.csv")

In [5]:
# Test: build features from a FEN + two ELOs
def features_from_fen(fen, white_elo, black_elo):
    board = chess.Board(fen)
    
    # Material
    PIECE_VALUES = {
        chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
        chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0,
    }
    mat_w = mat_b = 0
    for sq in chess.SQUARES:
        p = board.piece_at(sq)
        if p is None: continue
        if p.color == chess.WHITE:
            mat_w += PIECE_VALUES[p.piece_type]
        else:
            mat_b += PIECE_VALUES[p.piece_type]
    
    # Ply from FEN (fullmove number + whose turn)
    fullmove = board.fullmove_number
    ply = (fullmove - 1) * 2 + (0 if board.turn == chess.WHITE else 1)
    
    # Stockfish eval
    sf.set_fen_position(fen)
    ev = sf.get_evaluation()
    if ev["type"] == "cp":
        cp = ev["value"]
    elif ev["type"] == "mate":
        cp = 10000 if ev["value"] > 0 else -10000
    else:
        cp = 0
    if board.turn == chess.BLACK:
        cp = -cp
    
    return {
        "material_balance": mat_w - mat_b,
        "white_bishops": len(board.pieces(chess.BISHOP, chess.WHITE)),
        "black_bishops": len(board.pieces(chess.BISHOP, chess.BLACK)),
        "total_pieces": len(board.piece_map()),
        "ply": ply,
        "elo_diff": white_elo - black_elo,
        "stockfish_eval": cp,
        "stockfish_vs_material": cp - (mat_w - mat_b) * 100,
    }

# Test with the Speelman position
test_fen = "r4r1k/1p1b3P/p2qpp1p/8/n2p1P2/1N1nB3/PPP3Q1/1K4RR b - - 1 27"
feats = features_from_fen(test_fen, 2615, 2610)
print("Features built from FEN:")
for k, v in feats.items():
    print(f"  {k}: {v}")

# Predict
X_new = pd.DataFrame([feats])[feature_cols_v2]
proba = clf_v2.predict_proba(X_new)[0]
pred_class = clf_v2.classes_[np.argmax(proba)]
label = {1: "White wins", 0: "Draw", -1: "Black wins"}[pred_class]
print()
print(f"Prediction: {label} ({max(proba):.1%})")

Features built from FEN:
  material_balance: -4
  white_bishops: 1
  black_bishops: 1
  total_pieces: 24
  ply: 53
  elo_diff: 5
  stockfish_eval: 10000
  stockfish_vs_material: 10400

Prediction: White wins (57.9%)


In [2]:
# ============================================================
# PRE-MATCH MODEL — Building the dataset
# ============================================================

import chess
import chess.pgn
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import os

print("Imports OK")
print("Files in current folder:")
for f in sorted(os.listdir(".")):
    if f.endswith(".pgn"):
        size_mb = os.path.getsize(f) / (1024*1024)
        print(f"  {f} — {size_mb:.1f} MB")

Imports OK
Files in current folder:
  games.pgn — 0.0 MB
  mega2600_part_01.pgn — 25.0 MB
  mega2600_part_02.pgn — 25.0 MB
  mega2600_part_03.pgn — 25.0 MB
  mega2600_part_04.pgn — 25.0 MB
  mega2600_part_05.pgn — 7.5 MB


In [3]:
# ============================================================
# Load all games from the 5 PGN files, extract pre-match data
# ============================================================

pgn_files = [
    "mega2600_part_01.pgn",
    "mega2600_part_02.pgn",
    "mega2600_part_03.pgn",
    "mega2600_part_04.pgn",
    "mega2600_part_05.pgn",
]

MAX_PER_FILE = 1000   # cap per file to keep it manageable

rows = []
print(f"Loading up to {MAX_PER_FILE} games from each file...\n")

for pgn_file in pgn_files:
    if not os.path.exists(pgn_file):
        print(f"  ⚠️ Skipping {pgn_file} (not found)")
        continue

    count = 0
    with open(pgn_file, encoding="utf-8", errors="ignore") as f:
        while count < MAX_PER_FILE:
            game = chess.pgn.read_game(f)
            if game is None:
                break

            headers = game.headers
            result_str = headers.get("Result", "*")
            if result_str == "1-0":
                result = 1
            elif result_str == "0-1":
                result = -1
            elif result_str == "1/2-1/2":
                result = 0
            else:
                continue

            try:
                white_elo = int(headers.get("WhiteElo", 0) or 0)
                black_elo = int(headers.get("BlackElo", 0) or 0)
            except ValueError:
                continue

            # Skip games without valid ELOs
            if white_elo < 1000 or black_elo < 1000:
                continue

            rows.append({
                "white": headers.get("White", "?"),
                "black": headers.get("Black", "?"),
                "white_elo": white_elo,
                "black_elo": black_elo,
                "elo_diff": white_elo - black_elo,
                "avg_elo": (white_elo + black_elo) / 2,
                "white_advantage": 1,
                "result": result,
            })
            count += 1

    print(f"  {pgn_file}: loaded {count} games")

prematch_df = pd.DataFrame(rows)
print(f"\nTotal games: {len(prematch_df)}")
print(f"Shape: {prematch_df.shape}")
print()
print(prematch_df.head(10))

Loading up to 1000 games from each file...

  mega2600_part_01.pgn: loaded 1000 games
  mega2600_part_02.pgn: loaded 1000 games
  mega2600_part_03.pgn: loaded 1000 games
  mega2600_part_04.pgn: loaded 1000 games
  mega2600_part_05.pgn: loaded 1000 games

Total games: 5000
Shape: (5000, 8)

              white             black  white_elo  black_elo  elo_diff  \
0    Adams, Michael   Jones, Gawain C       2715       2606       109   
1    Short, Nigel D     Howell, David       2687       2625        62   
2     Howell, David    Adams, Michael       2625       2715       -90   
3   Jones, Gawain C     Howell, David       2606       2625       -19   
4    Short, Nigel D    Adams, Michael       2687       2715       -28   
5    Short, Nigel D    Adams, Michael       2687       2715       -28   
6    Adams, Michael    Short, Nigel D       2715       2687        28   
7     Howell, David   Jones, Gawain C       2620       2655       -35   
8     Howell, David   Jones, Gawain C       2639    

In [4]:
print("Total games:", len(prematch_df))
print()
print("Result distribution:")
print(prematch_df["result"].value_counts())
print()
print("ELO stats:")
print(f"  White ELO: min={prematch_df['white_elo'].min()}, max={prematch_df['white_elo'].max()}, mean={prematch_df['white_elo'].mean():.0f}")
print(f"  Black ELO: min={prematch_df['black_elo'].min()}, max={prematch_df['black_elo'].max()}, mean={prematch_df['black_elo'].mean():.0f}")
print()
print("ELO gap stats:")
print(f"  Min:  {prematch_df['elo_diff'].min()}")
print(f"  Max:  {prematch_df['elo_diff'].max()}")
print(f"  Mean: {prematch_df['elo_diff'].mean():.1f}")
print()
print("Sanity check — does higher-rated player win more?")
prematch_df["higher_rated"] = np.where(prematch_df["elo_diff"] > 0, 1,
                              np.where(prematch_df["elo_diff"] < 0, -1, 0))
same = prematch_df[prematch_df["higher_rated"] != 0]
agree = (same["higher_rated"] == same["result"]).mean()
print(f"  Higher-rated player wins {agree:.1%} of the time")

Total games: 5000

Result distribution:
result
 0    2208
 1    1593
-1    1199
Name: count, dtype: int64

ELO stats:
  White ELO: min=2600, max=2882, mean=2683
  Black ELO: min=2600, max=2882, mean=2684

ELO gap stats:
  Min:  -227
  Max:  222
  Mean: -0.6

Sanity check — does higher-rated player win more?
  Higher-rated player wins 33.8% of the time


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Features for pre-match prediction
feature_cols_pm = [
    "elo_diff",
    "avg_elo",
    "white_advantage",
]

X = prematch_df[feature_cols_pm]
y = prematch_df["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)}")
print(f"Testing set:  {len(X_test)}")
print()

# Try two models and compare
print("=== Logistic Regression ===")
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, pred_lr):.1%}")
print()
print(classification_report(y_test, pred_lr,
                            target_names=["Black win", "Draw", "White win"]))

print("=== Random Forest ===")
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, pred_rf):.1%}")
print()
print(classification_report(y_test, pred_rf,
                            target_names=["Black win", "Draw", "White win"]))

Training set: 4000
Testing set:  1000

=== Logistic Regression ===
Accuracy: 45.9%

              precision    recall  f1-score   support

   Black win       0.75      0.03      0.05       240
        Draw       0.46      0.85      0.59       442
   White win       0.46      0.25      0.32       318

    accuracy                           0.46      1000
   macro avg       0.56      0.37      0.32      1000
weighted avg       0.53      0.46      0.38      1000

=== Random Forest ===
Accuracy: 44.7%

              precision    recall  f1-score   support

   Black win       0.41      0.03      0.05       240
        Draw       0.45      0.85      0.59       442
   White win       0.42      0.20      0.27       318

    accuracy                           0.45      1000
   macro avg       0.43      0.36      0.31      1000
weighted avg       0.43      0.45      0.36      1000



In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

rf_balanced = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    class_weight="balanced",   # <-- the key change
    random_state=42,
)
rf_balanced.fit(X_train, y_train)
pred_bal = rf_balanced.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, pred_bal):.1%}")
print()
print(classification_report(y_test, pred_bal,
                            target_names=["Black win", "Draw", "White win"]))

Accuracy: 38.9%

              precision    recall  f1-score   support

   Black win       0.29      0.40      0.34       240
        Draw       0.48      0.33      0.39       442
   White win       0.40      0.46      0.43       318

    accuracy                           0.39      1000
   macro avg       0.39      0.40      0.39      1000
weighted avg       0.41      0.39      0.39      1000



In [7]:
def predict_prematch(white_elo, black_elo, model=rf_balanced, feature_cols=None):
    if feature_cols is None:
        feature_cols = feature_cols_pm

    elo_diff = white_elo - black_elo
    avg_elo = (white_elo + black_elo) / 2

    # ---- Even matchup: predict Draw ----
    if abs(elo_diff) < 30:
        return {
            "label": "Draw",
            "confidence": 0.55,
            "p_white": 0.28, "p_draw": 0.55, "p_black": 0.17,
            "source": "heuristic (even matchup)",
        }

    # ---- Extreme ELO gap: override ----
    if elo_diff >= 400:
        return {
            "label": "White wins", "confidence": 0.92,
            "p_white": 0.92, "p_draw": 0.05, "p_black": 0.03,
            "source": "override (White much stronger)",
        }
    if elo_diff <= -400:
        return {
            "label": "Black wins", "confidence": 0.92,
            "p_white": 0.03, "p_draw": 0.05, "p_black": 0.92,
            "source": "override (Black much stronger)",
        }

    # ---- Otherwise: ask the balanced model ----
    X_new = pd.DataFrame([{
        "elo_diff": elo_diff,
        "avg_elo": avg_elo,
        "white_advantage": 1,
    }])[feature_cols]

    proba = model.predict_proba(X_new)[0]
    probs = dict(zip(model.classes_, proba))
    pred_class = model.classes_[np.argmax(proba)]
    label = {1: "White wins", 0: "Draw", -1: "Black wins"}[pred_class]

    return {
        "label": label,
        "confidence": max(proba),
        "p_white": probs.get(1, 0),
        "p_draw": probs.get(0, 0),
        "p_black": probs.get(-1, 0),
        "source": "ML model (balanced)",
    }


# Re-test
test_cases = [
    (2680, 2680, "Equal ELO 2680"),
    (2700, 2680, "White +20"),
    (2750, 2650, "White +100"),
    (2850, 2450, "White +400"),
    (2850, 1850, "White +1000"),
    (1200, 1900, "White -700"),
]

print(f"{'Scenario':25s} {'Prediction':12s} {'Conf':>6s}  Source")
print("-" * 75)
for w, b, label in test_cases:
    r = predict_prematch(w, b)
    print(f"{label:25s} {r['label']:12s} {r['confidence']:>5.1%}  [{r['source']}]")

Scenario                  Prediction     Conf  Source
---------------------------------------------------------------------------
Equal ELO 2680            Draw         55.0%  [heuristic (even matchup)]
White +20                 Draw         55.0%  [heuristic (even matchup)]
White +100                White wins   59.0%  [ML model (balanced)]
White +400                White wins   92.0%  [override (White much stronger)]
White +1000               White wins   92.0%  [override (White much stronger)]
White -700                Black wins   92.0%  [override (Black much stronger)]


In [8]:
# Compare both models on the same test cases
test = [(2680, 2680), (2750, 2650), (2800, 2600), (2850, 2450)]

print(f"{'ELO diff':>10s}  {'Unbalanced':>20s}  {'Balanced':>20s}")
print("-" * 55)
for w, b in test:
    X = pd.DataFrame([{"elo_diff": w-b, "avg_elo": (w+b)/2, "white_advantage": 1}])[feature_cols_pm]
    
    p_un = rf.predict_proba(X)[0]
    l_un = {1:"W", 0:"D", -1:"B"}[rf.classes_[np.argmax(p_un)]]
    
    p_ba = rf_balanced.predict_proba(X)[0]
    l_ba = {1:"W", 0:"D", -1:"B"}[rf_balanced.classes_[np.argmax(p_ba)]]
    
    print(f"{w-b:>+10d}  {l_un} ({max(p_un):.1%})".ljust(32) + f"  {l_ba} ({max(p_ba):.1%})")

  ELO diff            Unbalanced              Balanced
-------------------------------------------------------
        +0  D (45.2%)             B (37.9%)
      +100  W (51.8%)             W (59.0%)
      +200  W (71.7%)             W (81.4%)
      +400  W (45.0%)             W (49.0%)


In [1]:
# ============================================================
# RETRAIN POSITION MODEL ON ALL 5000 GAMES
# ============================================================

import chess
import chess.pgn
import pandas as pd
import numpy as np
import io
import time
from stockfish import Stockfish

STOCKFISH_PATH = r"E:\chess-predictor\stockfish\stockfish-windows-x86-64-universal.exe"
PGN_FILES = [
    "mega2600_part_01.pgn",
    "mega2600_part_02.pgn",
    "mega2600_part_03.pgn",
    "mega2600_part_04.pgn",
    "mega2600_part_05.pgn",
]
MAX_GAMES = 5000
SNAPSHOT_PLY = 80
SF_DEPTH = 15

sf = Stockfish(path=STOCKFISH_PATH, depth=SF_DEPTH, parameters={"Threads": 2, "Hash": 256})

print(f"Ready to process up to {MAX_GAMES} games from {len(PGN_FILES)} files")
print(f"Snapshot at ply {SNAPSHOT_PLY}, Stockfish depth {SF_DEPTH}")

Ready to process up to 5000 games from 5 files
Snapshot at ply 80, Stockfish depth 15


In [2]:
# ============================================================
# Extract features + Stockfish eval for all games
# ============================================================

PIECE_VALUES = {
    chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3,
    chess.ROOK: 5, chess.QUEEN: 9, chess.KING: 0,
}

def features_at_ply(board):
    mat_w = mat_b = 0
    for sq in chess.SQUARES:
        p = board.piece_at(sq)
        if p is None:
            continue
        if p.color == chess.WHITE:
            mat_w += PIECE_VALUES[p.piece_type]
        else:
            mat_b += PIECE_VALUES[p.piece_type]
    return {
        "material_balance": mat_w - mat_b,
        "white_bishops": len(board.pieces(chess.BISHOP, chess.WHITE)),
        "black_bishops": len(board.pieces(chess.BISHOP, chess.BLACK)),
        "total_pieces": len(board.piece_map()),
    }

def stockfish_eval_from_fen(fen):
    sf.set_fen_position(fen)
    ev = sf.get_evaluation()
    if ev["type"] == "cp":
        cp = ev["value"]
    elif ev["type"] == "mate":
        cp = 10000 if ev["value"] > 0 else -10000
    else:
        cp = 0
    board_turn = fen.split()[1]
    if board_turn == "b":
        cp = -cp
    return cp

rows = []
start_time = time.time()
total_loaded = 0

for pgn_file in PGN_FILES:
    file_count = 0
    try:
        with open(pgn_file, encoding="utf-8", errors="ignore") as f:
            while total_loaded < MAX_GAMES:
                game = chess.pgn.read_game(f)
                if game is None:
                    break
                headers = game.headers
                result_str = headers.get("Result", "*")
                if result_str == "1-0":
                    result = 1
                elif result_str == "0-1":
                    result = -1
                elif result_str == "1/2-1/2":
                    result = 0
                else:
                    continue

                try:
                    white_elo = int(headers.get("WhiteElo", 0) or 0)
                    black_elo = int(headers.get("BlackElo", 0) or 0)
                except ValueError:
                    continue
                if white_elo < 1000 or black_elo < 1000:
                    continue

                board = game.board()
                moves = list(game.mainline_moves())
                for j, mv in enumerate(moves):
                    if j >= SNAPSHOT_PLY:
                        break
                    board.push(mv)

                feats = features_at_ply(board)
                feats["ply"] = min(SNAPSHOT_PLY, len(moves))
                feats["white_elo"] = white_elo
                feats["black_elo"] = black_elo
                feats["elo_diff"] = white_elo - black_elo
                feats["result"] = result

                cp = stockfish_eval_from_fen(board.fen())
                feats["stockfish_eval"] = cp
                feats["stockfish_vs_material"] = cp - feats["material_balance"] * 100

                rows.append(feats)
                total_loaded += 1
                file_count += 1

                if total_loaded % 250 == 0:
                    elapsed = time.time() - start_time
                    print(f"  {total_loaded} games done — {elapsed:.0f}s elapsed")

        print(f"  {pgn_file}: {file_count} games")
    except FileNotFoundError:
        print(f"  {pgn_file}: NOT FOUND — skipping")

print(f"\nTotal games: {len(rows)}")
print(f"Total time: {time.time() - start_time:.0f}s")

features_df_v2 = pd.DataFrame(rows)
print(f"Shape: {features_df_v2.shape}")
print(features_df_v2.head())

features_df_v2.to_csv("features_df_v2_5000.csv", index=False)
print("\n✅ Saved to features_df_v2_5000.csv")

  250 games done — 73s elapsed
  500 games done — 115s elapsed
  750 games done — 168s elapsed
  1000 games done — 222s elapsed
  1250 games done — 266s elapsed
  1500 games done — 314s elapsed
  1750 games done — 380s elapsed
  2000 games done — 430s elapsed
  2250 games done — 475s elapsed
  2500 games done — 538s elapsed
  2750 games done — 594s elapsed
  3000 games done — 636s elapsed
  3250 games done — 686s elapsed
  3500 games done — 737s elapsed
  3750 games done — 788s elapsed
  4000 games done — 835s elapsed
  4250 games done — 876s elapsed
  4500 games done — 931s elapsed
  4750 games done — 983s elapsed
  5000 games done — 1036s elapsed
  mega2600_part_01.pgn: 5000 games
  mega2600_part_02.pgn: 0 games
  mega2600_part_03.pgn: 0 games
  mega2600_part_04.pgn: 0 games
  mega2600_part_05.pgn: 0 games

Total games: 5000
Total time: 1036s
Shape: (5000, 11)
   material_balance  white_bishops  black_bishops  total_pieces  ply  \
0                 0              0              0    

In [4]:
# ============================================================
# Train Random Forest on 5000 games
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

feature_cols_v3 = [
    "material_balance",
    "white_bishops",
    "black_bishops",
    "total_pieces",
    "ply",
    "elo_diff",
    "stockfish_eval",
    "stockfish_vs_material",
]

X = features_df_v2[feature_cols_v3]
y = features_df_v2["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)}")
print(f"Testing set:  {len(X_test)}")
print()

clf_v3 = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
)
clf_v3.fit(X_train, y_train)

predictions = clf_v3.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"🎯 Test accuracy: {accuracy:.1%}")
print()
print(classification_report(y_test, predictions,
                            target_names=["Black win", "Draw", "White win"]))

# Save the model
joblib.dump(clf_v3, "chess_model.pkl")
joblib.dump(feature_cols_v3, "feature_cols.pkl")
print("\n✅ Model saved: chess_model.pkl")

Training set: 4000
Testing set:  1000

🎯 Test accuracy: 89.0%

              precision    recall  f1-score   support

   Black win       0.90      0.76      0.83       153
        Draw       0.89      0.93      0.91       579
   White win       0.89      0.88      0.88       268

    accuracy                           0.89      1000
   macro avg       0.89      0.86      0.87      1000
weighted avg       0.89      0.89      0.89      1000


✅ Model saved: chess_model.pkl


In [1]:
# ============================================================
# NEW FEATURE EXTRACTION FUNCTIONS
# ============================================================

def count_mobility(board, color):
    """Number of legal moves for the given color in this position."""
    b = board.copy()
    b.turn = color
    b.ep_square = None   # avoid invalid ep for the other side
    try:
        return sum(1 for _ in b.generate_legal_moves())
    except Exception:
        return 0

def king_pawn_shield(board, color):
    """Count friendly pawns in the 2 ranks in front of the king."""
    king_sq = board.king(color)
    if king_sq is None:
        return 0
    file = chess.square_file(king_sq)
    rank = chess.square_rank(king_sq)
    shields = 0
    for df in (-1, 0, 1):
        f = file + df
        if f < 0 or f > 7:
            continue
        for dr in (1, 2):
            r = rank + dr if color == chess.WHITE else rank - dr
            if r < 0 or r > 7:
                continue
            sq = chess.square(f, r)
            p = board.piece_at(sq)
            if p and p.piece_type == chess.PAWN and p.color == color:
                shields += 1
                break
    return shields

def has_bishop_pair(board, color):
    return 1 if len(board.pieces(chess.BISHOP, color)) >= 2 else 0

def pawn_issues(board, color):
    """Return (doubled_count, isolated_count) for the given color."""
    file_counts = [0] * 8
    for sq in chess.SQUARES:
        p = board.piece_at(sq)
        if p and p.piece_type == chess.PAWN and p.color == color:
            file_counts[chess.square_file(sq)] += 1
    doubled = sum(1 for c in file_counts if c >= 2)
    isolated = 0
    for f, c in enumerate(file_counts):
        if c == 0:
            continue
        left = file_counts[f-1] if f > 0 else 0
        right = file_counts[f+1] if f < 7 else 0
        if left == 0 and right == 0:
            isolated += c
    return doubled, isolated

def center_pawns(board, color):
    """Pawns on c, d, e, f files."""
    count = 0
    for f in (2, 3, 4, 5):
        for r in range(8):
            sq = chess.square(f, r)
            p = board.piece_at(sq)
            if p and p.piece_type == chess.PAWN and p.color == color:
                count += 1
    return count

def compute_new_features(board):
    """Return dict with 6 new features for this position."""
    w_mob = count_mobility(board, chess.WHITE)
    b_mob = count_mobility(board, chess.BLACK)
    w_doubled, w_isolated = pawn_issues(board, chess.WHITE)
    b_doubled, b_isolated = pawn_issues(board, chess.BLACK)

    return {
        "mobility_diff":       w_mob - b_mob,
        "king_safety_diff":    king_pawn_shield(board, chess.WHITE) - king_pawn_shield(board, chess.BLACK),
        "bishop_pair_diff":    has_bishop_pair(board, chess.WHITE) - has_bishop_pair(board, chess.BLACK),
        "doubled_pawns_diff":  b_doubled - w_doubled,
        "isolated_pawns_diff": b_isolated - w_isolated,
        "center_pawns_diff":   center_pawns(board, chess.WHITE) - center_pawns(board, chess.BLACK),
    }

print("✅ Feature functions defined")

✅ Feature functions defined


In [2]:
# ============================================================
# Re-read PGNs and compute the 6 new features for all 5000 games
# ============================================================

SNAPSHOT_PLY = 80
MAX_GAMES = 5000

new_rows = []
total_loaded = 0
start_time = time.time()

for pgn_file in PGN_FILES:
    try:
        with open(pgn_file, encoding="utf-8", errors="ignore") as f:
            while total_loaded < MAX_GAMES:
                game = chess.pgn.read_game(f)
                if game is None:
                    break
                headers = game.headers
                result_str = headers.get("Result", "*")
                if result_str not in ("1-0", "0-1", "1/2-1/2"):
                    continue
                try:
                    white_elo = int(headers.get("WhiteElo", 0) or 0)
                    black_elo = int(headers.get("BlackElo", 0) or 0)
                except ValueError:
                    continue
                if white_elo < 1000 or black_elo < 1000:
                    continue

                board = game.board()
                moves = list(game.mainline_moves())
                for j, mv in enumerate(moves):
                    if j >= SNAPSHOT_PLY:
                        break
                    board.push(mv)

                new_rows.append(compute_new_features(board))
                total_loaded += 1

                if total_loaded % 1000 == 0:
                    print(f"  {total_loaded} games done — {time.time()-start_time:.0f}s")
    except FileNotFoundError:
        pass

print(f"\nTotal: {len(new_rows)} games in {time.time()-start_time:.0f}s")

new_features_df = pd.DataFrame(new_rows)
print(f"Shape: {new_features_df.shape}")
print(new_features_df.head())

NameError: name 'time' is not defined

In [3]:
import chess
import chess.pgn
import pandas as pd
import numpy as np
import io
import time
from stockfish import Stockfish

PGN_FILES = [
    "mega2600_part_01.pgn",
    "mega2600_part_02.pgn",
    "mega2600_part_03.pgn",
    "mega2600_part_04.pgn",
    "mega2600_part_05.pgn",
]

print("✅ Imports done")

✅ Imports done


In [4]:
# ============================================================
# NEW FEATURE EXTRACTION FUNCTIONS
# ============================================================

def count_mobility(board, color):
    """Number of legal moves for the given color in this position."""
    b = board.copy()
    b.turn = color
    b.ep_square = None   # avoid invalid ep for the other side
    try:
        return sum(1 for _ in b.generate_legal_moves())
    except Exception:
        return 0

def king_pawn_shield(board, color):
    """Count friendly pawns in the 2 ranks in front of the king."""
    king_sq = board.king(color)
    if king_sq is None:
        return 0
    file = chess.square_file(king_sq)
    rank = chess.square_rank(king_sq)
    shields = 0
    for df in (-1, 0, 1):
        f = file + df
        if f < 0 or f > 7:
            continue
        for dr in (1, 2):
            r = rank + dr if color == chess.WHITE else rank - dr
            if r < 0 or r > 7:
                continue
            sq = chess.square(f, r)
            p = board.piece_at(sq)
            if p and p.piece_type == chess.PAWN and p.color == color:
                shields += 1
                break
    return shields

def has_bishop_pair(board, color):
    return 1 if len(board.pieces(chess.BISHOP, color)) >= 2 else 0

def pawn_issues(board, color):
    """Return (doubled_count, isolated_count) for the given color."""
    file_counts = [0] * 8
    for sq in chess.SQUARES:
        p = board.piece_at(sq)
        if p and p.piece_type == chess.PAWN and p.color == color:
            file_counts[chess.square_file(sq)] += 1
    doubled = sum(1 for c in file_counts if c >= 2)
    isolated = 0
    for f, c in enumerate(file_counts):
        if c == 0:
            continue
        left = file_counts[f-1] if f > 0 else 0
        right = file_counts[f+1] if f < 7 else 0
        if left == 0 and right == 0:
            isolated += c
    return doubled, isolated

def center_pawns(board, color):
    """Pawns on c, d, e, f files."""
    count = 0
    for f in (2, 3, 4, 5):
        for r in range(8):
            sq = chess.square(f, r)
            p = board.piece_at(sq)
            if p and p.piece_type == chess.PAWN and p.color == color:
                count += 1
    return count

def compute_new_features(board):
    """Return dict with 6 new features for this position."""
    w_mob = count_mobility(board, chess.WHITE)
    b_mob = count_mobility(board, chess.BLACK)
    w_doubled, w_isolated = pawn_issues(board, chess.WHITE)
    b_doubled, b_isolated = pawn_issues(board, chess.BLACK)

    return {
        "mobility_diff":       w_mob - b_mob,
        "king_safety_diff":    king_pawn_shield(board, chess.WHITE) - king_pawn_shield(board, chess.BLACK),
        "bishop_pair_diff":    has_bishop_pair(board, chess.WHITE) - has_bishop_pair(board, chess.BLACK),
        "doubled_pawns_diff":  b_doubled - w_doubled,
        "isolated_pawns_diff": b_isolated - w_isolated,
        "center_pawns_diff":   center_pawns(board, chess.WHITE) - center_pawns(board, chess.BLACK),
    }

print("✅ Feature functions defined")

✅ Feature functions defined


In [5]:
# ============================================================
# Re-read PGNs and compute the 6 new features for all 5000 games
# ============================================================

SNAPSHOT_PLY = 80
MAX_GAMES = 5000

new_rows = []
total_loaded = 0
start_time = time.time()

for pgn_file in PGN_FILES:
    try:
        with open(pgn_file, encoding="utf-8", errors="ignore") as f:
            while total_loaded < MAX_GAMES:
                game = chess.pgn.read_game(f)
                if game is None:
                    break
                headers = game.headers
                result_str = headers.get("Result", "*")
                if result_str not in ("1-0", "0-1", "1/2-1/2"):
                    continue
                try:
                    white_elo = int(headers.get("WhiteElo", 0) or 0)
                    black_elo = int(headers.get("BlackElo", 0) or 0)
                except ValueError:
                    continue
                if white_elo < 1000 or black_elo < 1000:
                    continue

                board = game.board()
                moves = list(game.mainline_moves())
                for j, mv in enumerate(moves):
                    if j >= SNAPSHOT_PLY:
                        break
                    board.push(mv)

                new_rows.append(compute_new_features(board))
                total_loaded += 1

                if total_loaded % 1000 == 0:
                    print(f"  {total_loaded} games done — {time.time()-start_time:.0f}s")
    except FileNotFoundError:
        pass

print(f"\nTotal: {len(new_rows)} games in {time.time()-start_time:.0f}s")

new_features_df = pd.DataFrame(new_rows)
print(f"Shape: {new_features_df.shape}")
print(new_features_df.head())

  1000 games done — 3s
  2000 games done — 6s
  3000 games done — 8s
  4000 games done — 11s
  5000 games done — 14s

Total: 5000 games in 14s
Shape: (5000, 6)
   mobility_diff  king_safety_diff  bishop_pair_diff  doubled_pawns_diff  \
0              0                 1                 0                   0   
1             -6                -2                -1                   0   
2            -22                -1                 0                   0   
3             25                 0                 0                   1   
4             -9                -1                 0                   0   

   isolated_pawns_diff  center_pawns_diff  
0                   -1                  0  
1                    0                  0  
2                    2                  0  
3                    1                  2  
4                   -3                  1  


In [6]:
# ============================================================
# Merge old features with new ones
# ============================================================

# Load the old features (8 original)
old_features = pd.read_csv("features_df_v2_5000.csv")
print(f"Old features: {old_features.shape}")

# Merge with new (both have 5000 rows, aligned by index)
merged_df = pd.concat([old_features.reset_index(drop=True),
                       new_features_df.reset_index(drop=True)], axis=1)

print(f"Merged: {merged_df.shape}")
print()
print("All columns:")
for c in merged_df.columns:
    print(f"  {c}")

# Save backup
merged_df.to_csv("features_df_v3_5000_with_extras.csv", index=False)
print("\n✅ Saved to features_df_v3_5000_with_extras.csv")

Old features: (5000, 11)
Merged: (5000, 17)

All columns:
  material_balance
  white_bishops
  black_bishops
  total_pieces
  ply
  white_elo
  black_elo
  elo_diff
  result
  stockfish_eval
  stockfish_vs_material
  mobility_diff
  king_safety_diff
  bishop_pair_diff
  doubled_pawns_diff
  isolated_pawns_diff
  center_pawns_diff

✅ Saved to features_df_v3_5000_with_extras.csv


In [7]:
# ============================================================
# Retrain Random Forest with 14 features
# ============================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib
import time

# NEW: 14 features (8 old + 6 new)
feature_cols_v4 = [
    # Original 8
    "material_balance",
    "white_bishops",
    "black_bishops",
    "total_pieces",
    "ply",
    "elo_diff",
    "stockfish_eval",
    "stockfish_vs_material",
    # NEW 6
    "mobility_diff",
    "king_safety_diff",
    "bishop_pair_diff",
    "doubled_pawns_diff",
    "isolated_pawns_diff",
    "center_pawns_diff",
]

X = merged_df[feature_cols_v4]
y = merged_df["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Features: {len(feature_cols_v4)}")
print(f"Training set: {len(X_train)}")
print(f"Testing set:  {len(X_test)}")
print()

clf_v4 = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
)

t0 = time.time()
clf_v4.fit(X_train, y_train)
print(f"Training time: {time.time()-t0:.1f}s")

predictions = clf_v4.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"\n🎯 Test accuracy: {accuracy:.1%}")
print()
print(classification_report(y_test, predictions,
                            target_names=["Black win", "Draw", "White win"]))

Features: 14
Training set: 4000
Testing set:  1000

Training time: 0.6s

🎯 Test accuracy: 89.1%

              precision    recall  f1-score   support

   Black win       0.94      0.76      0.84       153
        Draw       0.88      0.94      0.91       579
   White win       0.89      0.87      0.88       268

    accuracy                           0.89      1000
   macro avg       0.90      0.85      0.87      1000
weighted avg       0.89      0.89      0.89      1000

